# 006 · Decorators

> **Syntax › Functions** · Python 3.14

A decorator is a function that takes a function and returns a replacement for it. The `@decorator` line above a `def` looks like a C# attribute, but it is something quite different: it **runs code** and can change what the function does.

**In this notebook you will learn:**

- how wrapping one function in another works, and why `@` is just syntactic sugar
- how to write logging and timing decorators
- why you should always use `functools.wraps`
- decorators that take arguments (decorator factories)
- the order in which stacked decorators apply
- class-based decorators, briefly
- the built-in decorators you will meet every day: `@staticmethod`, `@classmethod`, `@property`, `@functools.cache`, `@functools.lru_cache`

## C# mental model

| C# | Python |
|---|---|
| `[Obsolete]`, `[HttpGet]` attribute | `@decorator` |
| attributes are **passive metadata**, read via reflection | decorators are **executable code** that run when the `def` runs |
| behaviour needs a framework (ASP.NET filters, middleware, AOP libraries, DI interceptors) | behaviour is added directly by the decorator |
| `static` method, `{ get; }` property | `@staticmethod`, `@property` |
| `MemoryCache` / hand-written memoisation | `@functools.cache` |

The closest C# analogues for *what decorators do* are ASP.NET Core **middleware**, **action filters**, and AOP-style interceptors: code that runs around a call.

## Functions wrapping functions

Since functions are objects *(see Functions › 001 · Defining Functions)*, a function can take another function, define a wrapper around it, and return the wrapper. The wrapper uses `*args, **kwargs` to accept and forward any arguments *(see Functions › 003 · Args and Kwargs)*.

In [ ]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)   # call the original
        return result.upper()            # ...and change its result
    return wrapper


def greet(name):
    return f"hello, {name}"


loud_greet = shout(greet)    # manual decoration
print(greet("alice"))
print(loud_greet("alice"))

## The `@` syntax is sugar

Writing `@shout` above a `def` means exactly `greet = shout(greet)` right after the function is defined. The original name now points at the wrapper.

In [ ]:
@shout
def greet(name):
    return f"hello, {name}"


# equivalent to: greet = shout(greet)
print(greet("bob"))
print(greet.__name__)     # 'wrapper': the metadata was lost (fixed below)

Because a decorator is just code that runs when the `def` statement executes, it can do work **at definition time**. A common use is registration, which in C# you would do with an attribute plus reflection at startup:

```csharp
// C#: attribute is metadata; something else must find it via reflection
[Command("hello")]
public static void Hello() { ... }

var commands = typeof(Program).GetMethods()
    .Where(m => m.GetCustomAttribute<CommandAttribute>() is not null);
```

In [ ]:
COMMANDS = {}


def command(func):
    COMMANDS[func.__name__] = func    # runs once, when the def runs
    return func                       # return the original, unchanged


@command
def status():
    return "all systems go"


@command
def version():
    return "1.0.0"


print(list(COMMANDS))
print(COMMANDS["status"]())

## A logging decorator, with `functools.wraps`

The wrapper replaced `greet.__name__` and the docstring. `functools.wraps(func)` copies the name, docstring, module, annotations and more onto the wrapper, and stores the original as `__wrapped__`. **Always use it.**

In [ ]:
import functools


def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"-> {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper


@log_calls
def add(a, b):
    """Add two numbers."""
    return a + b


add(2, b=3)
print(add.__name__, "|", add.__doc__)

## A timing decorator

This is the textbook decorator. In C# you would reach for a `Stopwatch` inside the method, middleware, or an AOP tool; in Python the decorator is the cross-cutting concern, applied with a single line.

In [ ]:
import time


def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:                          # report even if func raises
            elapsed = (time.perf_counter() - start) * 1000
            print(f"{func.__name__} took {elapsed:.1f} ms")
    return wrapper


@timed
def slow_sum(n):
    time.sleep(0.05)
    return sum(range(n))


print(slow_sum(1_000))

## Decorators with arguments (factories)

`@retry(times=3)` first **calls** `retry(times=3)`, which must return the actual decorator. That gives three levels of nesting: factory → decorator → wrapper.

```python
@retry(times=3)
def f(): ...
# means:  f = retry(times=3)(f)
```

In [ ]:
def retry(times=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

In [ ]:
calls = {"count": 0}


@retry(times=3, exceptions=(ConnectionError,))
def flaky_fetch():
    calls["count"] += 1
    if calls["count"] < 3:
        raise ConnectionError("timeout")
    return "data"


print(flaky_fetch())

⚠️ **Gotcha:** forgetting the parentheses on a factory, `@retry` instead of `@retry()`, passes the function as `times`. Nothing fails at definition time; the error only shows up when you call the function.

In [ ]:
@retry                 # oops: retry(func) returns 'decorator', not a wrapper
def ping():
    return "pong"


print(ping)            # ping is now the inner 'decorator' function
try:
    ping()
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Stacking decorators

Decorators apply **bottom-up** (the one closest to the `def` wraps first), and the resulting wrappers then run **top-down** when called, like layers of middleware.

```python
@a
@b
def f(): ...
# means:  f = a(b(f))
```

In [ ]:
def tag(label):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"enter {label}")
            result = func(*args, **kwargs)
            print(f"exit  {label}")
            return result
        return wrapper
    return decorator


@tag("outer")
@tag("inner")
def work():
    print("  working")


work()

Order matters in practice: `@timed` above `@retry(...)` times all attempts together, while `@timed` below `@retry(...)` times each attempt separately.

## Class-based decorators

Any **callable** can be a decorator, including a class. `__init__` receives the function and `__call__` runs on each call *(see Classes & OOP › 007 · Dunder Methods)*. This is handy when the decorator needs state you want to inspect.

In [ ]:
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # the class version of @wraps
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)


@CountCalls
def say(word):
    return word


say("a")
say("b")
print(say.calls, say.__name__)

## Built-in decorators

You will use these constantly. The class-related ones are covered in depth in *Classes & OOP › 002 · Class and Static Members* and *Classes & OOP › 003 · Properties*.

| Decorator | Purpose | C# |
|---|---|---|
| `@staticmethod` | method without `self` | `static` method |
| `@classmethod` | method receiving the class as `cls` | `static` method (factory), no direct equivalent |
| `@property` | computed attribute | property `{ get; }` |
| `@functools.cache` | unbounded memoisation | hand-written dictionary cache |
| `@functools.lru_cache(maxsize=n)` | bounded memoisation | `MemoryCache` with size limit |

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):              # accessed like a field: t.fahrenheit
        return self.celsius * 9 / 5 + 32

    @classmethod
    def from_fahrenheit(cls, f):       # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_valid(celsius):             # no self, no cls
        return celsius >= -273.15


t = Temperature.from_fahrenheit(212)
print(t.celsius, t.fahrenheit, Temperature.is_valid(-300))

`@functools.cache` memoises a function: results are stored by argument, so repeated calls are free. Arguments must be hashable. `@lru_cache(maxsize=...)` does the same with a size limit.

In [ ]:
@functools.cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(80))            # instant; uncached this recursion would never finish
print(fib.cache_info())


@functools.lru_cache(maxsize=2)
def lookup(key):
    print(f"  computing {key}")
    return key.upper()


for key in ["a", "b", "a", "c", "a"]:
    lookup(key)
print(lookup.cache_info())

⚠️ **Gotcha:** `@cache` on an instance method keeps every `self` alive forever (the cache holds a reference to it), and it never expires. Use it for pure functions, and consider `functools.cached_property` for per-instance computed values.

## Summary

| Concept | C# | Python |
|---|---|---|
| Annotate a method | `[Attribute]` (metadata only) | `@decorator` (runs code) |
| What `@d` means | n/a | `f = d(f)` |
| Wrap behaviour around calls | middleware, filters, interceptors | wrapper function in a decorator |
| Preserve name/docs | n/a | `@functools.wraps(func)` |
| Parameterised | attribute constructor args | factory: `@retry(times=3)` |
| Stacking | order usually irrelevant | bottom-up application: `a(b(f))` |
| Static / class method | `static` | `@staticmethod` / `@classmethod` |
| Property | `{ get; }` | `@property` |
| Memoisation | manual cache | `@functools.cache` / `@lru_cache` |

## Exercises

1. Write a decorator `@debug` that prints the function name, its arguments and its return value. Use `functools.wraps`.
2. Write a decorator factory `@repeat(n)` that calls the function `n` times and returns a list of the results.
3. Write `@validate_positive` that raises `ValueError` if any positional argument is a negative number, then apply it to `def area(w, h): return w * h`.
4. In what order are the messages printed by `@tag("A")` `@tag("B")` `@tag("C")` on a function, using `tag` from this notebook?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import functools

# 1
def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{func.__name__}{args} {kwargs} -> {result!r}")
        return result
    return wrapper

# 2
def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(n)]
        return wrapper
    return decorator

# 3
def validate_positive(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if any(isinstance(a, (int, float)) and a < 0 for a in args):
            raise ValueError(f"{func.__name__}: negative argument in {args}")
        return func(*args, **kwargs)
    return wrapper

@validate_positive
def area(w, h):
    return w * h

# 4 — enter A, enter B, enter C, (body), exit C, exit B, exit A.
#     f = A(B(C(f))): C is applied first, but A's wrapper runs first.
```

</details>